## 1. Tiền xử lý 


In [2]:
from pathlib import Path
import sys

project = Path.cwd()
if project.name == "notebooks":
    project = project.parent
sys.path.insert(0, str(project))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.evaluation import DATA, RESULTS, SEED, load_data, make_folds, save_submission
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from src.features import add_features, preprocess

X, y_log, X_test, test_ids, sample = load_data()
processor = preprocess(scale=False)


## 2. Tạo đặc trưng

In [3]:
feature_sets = {
    "basic": (X, X_test),
    "engineered": (add_features(X), add_features(X_test)),
}
folds = list(make_folds().split(X))


## 3. Huấn luyện và tuning

In [5]:
from sklearn.ensemble import RandomForestRegressor

name = "random_forest"
pipeline = Pipeline([("preprocess", processor), ("model", RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=2))])
grid = {"model__min_samples_leaf": [1, 3, 5]}
scoring = "neg_root_mean_squared_error"

# So sánh đặc trưng với cùng mô hình và tham số.
ablation = {
    label: -cross_val_score(pipeline, train_features, y_log, cv=folds, scoring=scoring)
    for label, (train_features, _) in feature_sets.items()
}
feature_choice = min(ablation, key=lambda label: ablation[label].mean())
train_features, test_features = feature_sets[feature_choice]
search = GridSearchCV(pipeline, grid, cv=folds, scoring=scoring, error_score="raise")
search.fit(train_features, y_log)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__min_samples_leaf': [1, 3, ...]}"
,scoring,'neg_root_mean_squared_error'
,n_jobs,None
,refit,True
,cv,"[(array([ 0, ...shape=(1168,)), ...), (array([ 0, ...shape=(1168,)), ...), ...]"
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,'raise'
,return_train_score,False
,transformers,"[('numeric', ...), ('categorical', ...)]"



### Nhận xét về quá trình huấn luyện Random Forest

- Mô hình sử dụng: Áp dụng thuật toán `RandomForestRegressor` với 200 cây quyết định để dự đoán giá bán nhà trên thang đo logarit.
- Kiểm định chéo: Sử dụng phương pháp kiểm định chéo 5 phần nhằm đánh giá khả năng dự đoán và hạn chế sự phụ thuộc vào một lần chia dữ liệu.
- So sánh đặc trưng: Thử nghiệm hai nhóm đặc trưng `basic` và `engineered`, sau đó lựa chọn nhóm có sai số trung bình thấp hơn.
- Tối ưu siêu tham số: Sử dụng `GridSearchCV` để thử nghiệm các giá trị `min_samples_leaf` gồm 1, 3 và 5, nhằm điều chỉnh độ phức tạp của các cây quyết định và hạn chế quá khớp.
- Chỉ số đánh giá: Sử dụng `RMSE` trên thang đo logarit để đo lường mức độ sai lệch giữa giá trị dự đoán và giá trị thực tế.

**Kết luận:** Quá trình huấn luyện giúp lựa chọn nhóm đặc trưng và cấu hình Random Forest phù hợp nhất trong các phương án thử nghiệm. Kết quả cụ thể được phân tích ở phần đánh giá mô hình.


## 4. Đánh giá và xuất CSV

In [6]:
import json

rows = []
for variant, scores in ablation.items():
    rows.append({"model": name, "variant": variant, "features": variant, "params": "default", **{
        f"fold_{i}": score for i, score in enumerate(scores, 1)
    }})
tuned_scores = [-search.cv_results_[f"split{i}_test_score"][search.best_index_] for i in range(len(folds))]
rows.append({"model": name, "variant": "tuned", "features": feature_choice,
             "params": json.dumps(search.best_params_), **{
                 f"fold_{i}": score for i, score in enumerate(tuned_scores, 1)
             }})
scores = pd.DataFrame(rows)
fold_columns = [f"fold_{i}" for i in range(1, len(folds) + 1)]
scores["cv_mean"] = scores[fold_columns].mean(axis=1)
scores["cv_std"] = scores[fold_columns].std(axis=1, ddof=0)
scores["selected"] = False
best = scores["cv_mean"].idxmin()
scores.loc[best, "selected"] = True
selected = scores.loc[best]
if selected["variant"] == "tuned":
    final_model = search.best_estimator_
else:
    train_features, test_features = feature_sets[selected["features"]]
    final_model = clone(pipeline).fit(train_features, y_log)
submission = save_submission(name, final_model.predict(test_features), test_ids, sample)
cv_dir = RESULTS / "cv"
cv_dir.mkdir(parents=True, exist_ok=True)
scores.to_csv(cv_dir / f"{name}.csv", index=False)
display(scores)
display(submission.head())


,model,variant,features,params,fold_1,fold_2,fold_3,fold_4,fold_5,cv_mean,cv_std,selected
0,random_forest,basic,basic,default,0.145389,0.127150,0.176833,0.148474,0.122956,0.144160,0.019113,False
1,random_forest,engineered,engineered,default,0.145292,0.126946,0.175994,0.144443,0.122762,0.143087,0.018779,False
2,random_forest,tuned,engineered,"{""model__min_samples_leaf"": 3}",0.147232,0.126427,0.172780,0.146338,0.122365,0.143028,0.017980,True


,Id,SalePrice
0,1461,125813.335877
1,1462,154632.340880
2,1463,175253.379449
3,1464,182128.681071
4,1465,195227.105506


### Nhận xét kết quả đánh giá và xuất CSV

- Kết quả kiểm định chéo 5 phần cho thấy cả ba phiên bản Random Forest đều có mức sai số tương đối gần nhau. Phiên bản Basic đạt RMSE trung bình 0,144160, trong khi phiên bản Engineered giảm xuống 0,143087. Điều này cho thấy việc bổ sung các đặc trưng mới đã góp phần cải thiện khả năng dự đoán của mô hình.

- Sau quá trình tinh chỉnh siêu tham số bằng GridSearchCV, phiên bản Tuned đạt RMSE trung bình thấp nhất là 0,143028, với độ lệch chuẩn 0,017980. So với phiên bản Basic, sai số trung bình giảm khoảng 0,79%. Tuy nhiên, mức cải thiện so với phiên bản Engineered không đáng kể, cho thấy việc tinh chỉnh siêu tham số chỉ mang lại lợi ích nhỏ trong phạm vi thử nghiệm.

- Dựa trên kết quả kiểm định chéo, phiên bản Tuned được lựa chọn để dự đoán giá nhà trên tập dữ liệu kiểm tra. Giá trị dự đoán được chuyển từ thang logarit về giá bán ban đầu và xuất thành file CSV gồm hai cột `Id` và `SalePrice`, đáp ứng định dạng của bài toán House Prices.

**Kết luận:** Random Forest sau tinh chỉnh đạt kết quả tốt nhất trong ba phiên bản được thử nghiệm. Tuy nhiên, cần tiếp tục mở rộng phạm vi tinh chỉnh và đối chiếu với các mô hình khác để đánh giá khả năng cải thiện hiệu suất dự đoán.